In [1]:
# Export random sample of Florida permits for data exploration

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np

rng = np.random.RandomState(42)

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

SUMMARY_FILENAME = "dewey_summary"
SUMMARY_FILEPATH = os.path.join(MY_DATA_PATH, f"{SUMMARY_FILENAME}.parquet")

COLUMNS = [
    'PERMIT_NUMBER', 'JURISDICTION', 'STATE', 'FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE', 'STATUS_NORMALIZED', 'STATUS_ORIGINAL', 'RECORD_TYPE_ORIGINAL', 'RECORD_SUBTYPE_ORIGINAL', 'APN', 'STREET', 'ZIPCODE', 'DATA', 'RESIDENTIAL'
]  # Columns to load from data file

TARGET_RECORDS_PER_CITY = 2000
STATE = "TX"
OUTPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data", "permits_tx_sample.parquet")


In [2]:
# Load the data

summ_df = pd.read_parquet(SUMMARY_FILEPATH)
summ_df = summ_df.groupby(['STATE', 'JURISDICTION']).agg(COUNT = ('COUNT', 'sum')).reset_index()
summ_df = summ_df.loc[summ_df['STATE']==STATE].sort_values(by='COUNT', ascending=False).reset_index(drop=True)


In [3]:
# Retrieve cities data

jurisdictions = summ_df['JURISDICTION'].tolist()
states = summ_df['STATE'].tolist()

df = []
for j, s in zip(jurisdictions, states):
    mydf = du.get_data_for_jurisdiction(j, s, columns=COLUMNS, n_records=TARGET_RECORDS_PER_CITY, rng=rng)
    df.append(mydf)

df = pd.concat(df)

print("\nDone!\n")
print(df["JURISDICTION"].value_counts())


Retrieving data for Austin TX ... 135/135 files ... elapsed time 61.81 seconds              
Retrieving data for Fort Worth TX ... 105/105 files ... elapsed time 52.45 seconds              
Retrieving data for Houston TX ... 27/27 files ... elapsed time 11.93 seconds              
Retrieving data for San Antonio TX ... 104/104 files ... elapsed time 49.06 seconds              
Retrieving data for Dallas TX ... 37/37 files ... elapsed time 18.33 seconds              
Retrieving data for Harris County TX ... 48/48 files ... elapsed time 24.19 seconds              
Retrieving data for El Paso TX ... 34/34 files ... elapsed time 15.83 seconds              
Retrieving data for Plano TX ... 14/14 files ... elapsed time 6.01 seconds              
Retrieving data for Frisco TX ... 33/33 files ... elapsed time 16.21 seconds              
Retrieving data for Pearland TX ... 19/19 files ... elapsed time 8.67 seconds              
Retrieving data for Grand Prairie TX ... 16/16 files ... elapsed ti

In [4]:
# Save data
df.to_parquet(OUTPUT_FILEPATH)
